# 02 — Search Landscapes & Geometry-Guided Search

Reproduces paper **Fig. 1** (`geo_limited_search_improvement.png`),
**Fig. 3** (`landscape_comparison_gemma3.png`), **Fig. 9**
(`TPE_vs_grid_search.png` in Appendix §F).

All inputs come from `results/optuna/` (TPE trial histories per seed and
search mode) and `results/steering_evaluations/` (the §3 grid-sweep CSVs).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grace.analysis.load_results import load_optuna_history, load_summary_results
from grace.analysis.t95 import t95

FIG_DIR = Path('Images'); FIG_DIR.mkdir(exist_ok=True)
GEMMA3 = 'google/gemma-3-27b-it'
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

## Fig. 3 — Landscape contrast: broad vs. narrow optimum (Gemma3)

Two example concepts — one with a broad, forgiving landscape (e.g. `professional`)
and one with a narrow, sharp peak (e.g. `golden_gate_centric`). Each panel is
the (layer × coef) heatmap of mean utility from the §3 grid sweep.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, concept in zip(axes, ['professional', 'golden_gate_centric']):
    rows = load_summary_results(GEMMA3, concept, method='pv')
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index='layer', columns='coef', values='mean_utility', aggfunc='mean').sort_index()
    im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='viridis',
                   extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()])
    ax.set_title(concept)
    ax.set_xlabel('coefficient'); ax.set_ylabel('layer')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(FIG_DIR / 'landscape_comparison_gemma3.png', dpi=150); plt.show()

## Fig. 1 — Geometry-restricted TPE vs. unconstrained TPE

Mean best-found-utility as a function of trial number, averaged across the
20 concepts on Gemma3-27B and the 3 seeds. Two curves: `unconstrained` (full
layer range) vs. `top15_pl` (alignment-restricted). Convergence is faster on
the alignment-restricted variant (paper's 44.5 % T95 reduction).


In [ ]:
def mean_best_curve(concepts, mode):
    histories = []
    for c in concepts:
        rows = load_optuna_history(GEMMA3, c, method='pv', mode=mode)
        if not rows:
            continue
        df = pd.DataFrame(rows)
        bv = df.groupby('trial')['best_value_so_far'].mean()
        histories.append(bv)
    if not histories:
        return pd.Series(dtype=float)
    big = pd.concat(histories, axis=1)
    return big.mean(axis=1)

uc = mean_best_curve(CONCEPTS, 'unconstrained')
tk = mean_best_curve(CONCEPTS, 'top15_pl')
plt.figure(figsize=(6, 4))
plt.plot(uc.index, uc.values, label='unconstrained TPE')
plt.plot(tk.index, tk.values, label='top-15 PL alignment')
plt.xlabel('TPE trial'); plt.ylabel('Mean best-found utility')
plt.title('Geometry-restricted search converges faster (Gemma3-27B)')
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / 'geo_limited_search_improvement.png', dpi=150); plt.show()

## Fig. 9 — TPE vs. fixed-interval grid search

TPE typically beats the §3 grid sweep within ~15 trials. We compute the
best-utility curve for unconstrained TPE and overlay the grid-search result
as a horizontal line per concept (averaged).


In [ ]:
tpe_curve = mean_best_curve(CONCEPTS, 'unconstrained')
grid_best = []
for c in CONCEPTS:
    rows = load_summary_results(GEMMA3, c, method='pv')
    if rows:
        grid_best.append(max(r['mean_utility'] for r in rows if r.get('mean_utility') is not None))
grid_mean = float(np.mean(grid_best)) if grid_best else float('nan')
plt.figure(figsize=(6, 4))
plt.plot(tpe_curve.index, tpe_curve.values, label='TPE (unconstrained)')
plt.axhline(grid_mean, color='gray', linestyle='--', label='grid sweep best')
plt.xlabel('TPE trial'); plt.ylabel('Mean best-found utility')
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / 'TPE_vs_grid_search.png', dpi=150); plt.show()